# Sentiment Analysis of Tunisian Dialect Comments

Three-class sentiment classification (**negative / neutre / positive**) of user comments written in
Tunisian Arabic about the Tunisian economic crisis. The notebook compares classical machine learning,
hybrid CNN + recurrent networks, and a fine-tuned transformer ([AraBERT](https://huggingface.co/aubmindlab/bert-base-arabertv02)),
all evaluated on the **same held-out test set**.

## Pipeline

| Step | What happens |
|---|---|
| 1. Load & clean labels | Read the Excel file, drop missing rows, normalise labels, drop the `lang` and `vide` rows |
| 2. Clean text | Keep Arabic-script characters only, drop empty and duplicate comments |
| 3. Split once | Stratified 80/20 train/test split, fixed seed. **Every vectoriser and embedding is fitted on the training set only** |
| 4. Features | TF-IDF (1000 features) and Word2Vec (100-d mean-pooled) |
| 5. Models | Naive Bayes, SVMs (linear / RBF / sigmoid / poly), MLP, Random Forest, Logistic Regression, CNN+LSTM, CNN+BiLSTM, CNN+GRU |
| 6. Transformer | AraBERT fine-tuned for 3 epochs (needs a GPU) |
| 7. Comparison | One table, ranked by **macro-F1** |

## Data

The dataset is **not included** in this repository. The notebook expects an Excel file with two columns:

| column | content |
|---|---|
| `comment` | raw comment text |
| `label` | `positive`, `negative`, `neutre` (rows labelled `lang` or `vide` are discarded) |

Set the location with the `DATA_PATH` environment variable or edit the config cell below.

> **Source and terms:** _describe where the comments were collected and under which terms before sharing the data._
> The comments are third-party user content: do not redistribute them unless the platform's terms allow it and
> the text has been anonymised.

## How to run

```bash
pip install -r requirements.txt
jupyter lab
```

The classical and Keras models run on CPU. The AraBERT section is slow without a GPU
(set `RUN_ARABERT = False` in the config cell to skip it). On Colab, choose a GPU runtime.

## Known limitations

* **Class imbalance:** negative comments are roughly 60 % of the data, so a model that always answers "negative"
  already gets about 60 % accuracy. That is why a majority-class baseline is included and why **macro-F1** is the main metric.
* **Arabic-only cleaning:** Tunisian dialect is often written in Latin letters (Arabizi) or mixed with French.
  Everything outside the Arabic script is removed, so that information is lost.
* **Single split, no hyper-parameter search:** differences of a point or two between models are within noise.

## 1. Setup

In [ ]:
import os
import random
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from gensim.models import Word2Vec
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC

# ---- Configuration ---------------------------------------------------------
SEED = 42
DATA_PATH = os.environ.get("DATA_PATH", "data/crise economique (labeled).xlsx")
TEST_SIZE = 0.2
RUN_KERAS = True      # CNN + LSTM / BiLSTM / GRU hybrids
RUN_ARABERT = True    # fine-tune AraBERT (needs a GPU to be practical)
# -----------------------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

## 2. Load and clean the data

In [ ]:
df = pd.read_excel(DATA_PATH)[["comment", "label"]].dropna()
df["label"] = df["label"].str.lower().str.strip()
df = df[~df["label"].isin(["lang", "vide"])].copy()

print("Label distribution after filtering:")
print(df["label"].value_counts())

In [ ]:
# Keep Arabic-script characters only (U+0600-U+06FF); everything else becomes a space.
# NOTE: this also removes Latin-script (Arabizi/French) text, digits and emojis.
NON_ARABIC = re.compile(r"[^\u0600-\u06FF]+")

def clean_text(text):
    return NON_ARABIC.sub(" ", str(text)).strip()

df["text"] = df["comment"].map(clean_text)

n_before = len(df)
df = df[df["text"] != ""]                      # comments with no Arabic text -> empty vectors / NaN embeddings
n_empty = n_before - len(df)
df = df.drop_duplicates(subset="text")         # identical texts could land in both train and test
n_dupes = n_before - n_empty - len(df)
df = df.reset_index(drop=True)

print(f"Removed {n_empty} comments with no Arabic text and {n_dupes} duplicate texts -> {len(df)} comments kept")
df[["text", "label"]].head()

## 3. Train / test split

The split is done **once**, before any feature extraction, so no statistic (vocabulary, IDF weights, word
embeddings) is learned from the test comments. All models below use this same split.

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df["label"])
CLASS_NAMES = list(label_encoder.classes_)

texts = df["text"].to_numpy()
train_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=TEST_SIZE, stratify=y, random_state=SEED
)
X_train_text, X_test_text = texts[train_idx], texts[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("Classes:", CLASS_NAMES)
print(f"Train: {len(y_train)}  |  Test: {len(y_test)}")

### Evaluation helpers

`evaluate` stores accuracy, macro-F1 and weighted-F1 for every model in one dictionary, so all approaches end up in
a single comparison table (section 8).

In [ ]:
RESULTS = {}

def evaluate(name, y_pred):
    y_pred = np.asarray(y_pred)
    RESULTS[name] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(y_test, y_pred, average="macro"),
        "weighted_f1": f1_score(y_test, y_pred, average="weighted"),
        "y_pred": y_pred,
    }
    r = RESULTS[name]
    print(f"{name:<40} accuracy={r['accuracy']:.3f}  macro-F1={r['macro_f1']:.3f}")

# Reference point: always predict the most frequent class
baseline = DummyClassifier(strategy="most_frequent").fit(X_train_text, y_train)
evaluate("Majority-class baseline", baseline.predict(X_test_text))

## 4. Classical models

The same set of classifiers is trained on two feature types: TF-IDF and Word2Vec.

In [ ]:
def classical_models():
    return {
        "Naive Bayes": GaussianNB(),
        "Linear SVM": SVC(kernel="linear"),
        "RBF SVM": SVC(kernel="rbf"),
        "Sigmoid SVM": SVC(kernel="sigmoid"),
        "Polynomial (deg 2) SVM": SVC(kernel="poly", degree=2),
        "Neural network (MLP)": MLPClassifier(
            hidden_layer_sizes=(100, 20), activation="logistic", solver="adam",
            max_iter=500, random_state=SEED,
        ),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=SEED),
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    }

def run_classical(feature_name, X_train, X_test):
    for model_name, model in classical_models().items():
        model.fit(X_train, y_train)
        evaluate(f"{feature_name} + {model_name}", model.predict(X_test))

### 4.1 TF-IDF features

The vectoriser is fitted on the training comments only.

In [ ]:
tfidf = TfidfVectorizer(max_features=1000, min_df=7, max_df=0.8)
X_train_tfidf = tfidf.fit_transform(X_train_text).toarray()
X_test_tfidf = tfidf.transform(X_test_text).toarray()
print("TF-IDF matrix:", X_train_tfidf.shape, "(train)", X_test_tfidf.shape, "(test)")

run_classical("TF-IDF", X_train_tfidf, X_test_tfidf)

### 4.2 Word2Vec features

Word2Vec is trained on the training comments only. Each comment is the mean of its word vectors
(a zero vector if none of its words is in the vocabulary), then standardised with a scaler fitted on the training
set: mean-pooled embeddings have very small values, and without scaling several models (linear SVM, MLP, logistic
regression, the Keras networks) tend to collapse to the majority class. Arabic stop-word removal is deliberately not applied:
standard Arabic stop-word lists contain negation words that matter for sentiment.

In [ ]:
def tokenize(text):
    return [w for w in text.split() if w.isalpha()]

train_tokens = [tokenize(t) for t in X_train_text]
test_tokens = [tokenize(t) for t in X_test_text]

w2v = Word2Vec(sentences=train_tokens, vector_size=100, window=5, min_count=1, workers=1, seed=SEED)

def embed(token_lists):
    out = np.zeros((len(token_lists), w2v.vector_size), dtype="float32")
    for i, tokens in enumerate(token_lists):
        vectors = [w2v.wv[t] for t in tokens if t in w2v.wv]
        if vectors:
            out[i] = np.mean(vectors, axis=0)
    return out

w2v_scaler = StandardScaler()
X_train_w2v = w2v_scaler.fit_transform(embed(train_tokens))
X_test_w2v = w2v_scaler.transform(embed(test_tokens))
print("Word2Vec matrix:", X_train_w2v.shape, "(train)", X_test_w2v.shape, "(test)")

run_classical("Word2Vec", X_train_w2v, X_test_w2v)

## 5. Hybrid CNN + recurrent networks

A 1-D convolution branch and a recurrent branch (LSTM, Bi-LSTM or GRU) read the same feature vector, and their
outputs are concatenated before the classification layers. The vector is treated as a sequence of length
`n_features` with one channel, so this is an experimental architecture rather than a conventional sequence model.

Training uses a 20 % validation split taken from the **training** set with early stopping; the test set is only
used for the final evaluation.

In [ ]:
if RUN_KERAS:
    import keras
    from keras import layers

    keras.utils.set_random_seed(SEED)

    RNN_LAYERS = {
        "LSTM": lambda: layers.LSTM(50),
        "Bi-LSTM": lambda: layers.Bidirectional(layers.LSTM(50)),
        "GRU": lambda: layers.GRU(50),
    }

    def build_hybrid(n_features, n_classes, rnn):
        inputs = layers.Input(shape=(n_features, 1))
        conv = layers.Conv1D(filters=32, kernel_size=5, activation="relu")(inputs)
        conv = layers.MaxPooling1D(pool_size=2)(conv)
        conv = layers.Flatten()(conv)
        recurrent = RNN_LAYERS[rnn]()(inputs)
        x = layers.concatenate([conv, recurrent])
        x = layers.Dense(50, activation="relu")(x)
        outputs = layers.Dense(n_classes, activation="softmax")(x)
        model = keras.Model(inputs, outputs)
        model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        return model

    def run_hybrid(feature_name, X_train, X_test, rnn):
        model = build_hybrid(X_train.shape[1], len(CLASS_NAMES), rnn)
        early_stop = keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True
        )
        model.fit(
            X_train[..., None], y_train,
            epochs=20, batch_size=32, validation_split=0.2,
            callbacks=[early_stop], verbose=0,
        )
        y_pred = model.predict(X_test[..., None], verbose=0).argmax(axis=1)
        evaluate(f"{feature_name} + CNN + {rnn}", y_pred)

    for feature_name, (Xtr, Xte) in {
        "TF-IDF": (X_train_tfidf, X_test_tfidf),
        "Word2Vec": (X_train_w2v, X_test_w2v),
    }.items():
        for rnn in RNN_LAYERS:
            run_hybrid(feature_name, Xtr, Xte, rnn)
else:
    print("Skipped (RUN_KERAS = False)")

## 6. AraBERT fine-tuning

The cleaned comments are tokenised with the AraBERT tokenizer (with attention masks), and the pre-trained model
is fine-tuned for a fixed number of epochs. The test set is never used to choose epochs or hyper-parameters.

In [ ]:
if RUN_ARABERT:
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        get_linear_schedule_with_warmup,
    )

    MODEL_NAME = "aubmindlab/bert-base-arabertv02"
    MAX_LEN, BATCH_SIZE, EPOCHS, LEARNING_RATE = 128, 16, 3, 2e-5

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cpu":
        print("WARNING: no GPU found, fine-tuning on CPU will be very slow.")

    torch.manual_seed(SEED)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(SEED)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def make_loader(text_array, labels, shuffle):
        enc = tokenizer(
            list(text_array), truncation=True, max_length=MAX_LEN,
            padding=True, return_tensors="pt",
        )
        dataset = TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(labels).long())
        return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle)

    train_loader = make_loader(X_train_text, y_train, shuffle=True)
    test_loader = make_loader(X_test_text, y_test, shuffle=False)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(CLASS_NAMES)
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
    )

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0
        for input_ids, attention_mask, labels in train_loader:
            input_ids, attention_mask, labels = (t.to(device) for t in (input_ids, attention_mask, labels))
            optimizer.zero_grad()
            loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
        print(f"Epoch {epoch + 1}/{EPOCHS} - mean training loss: {total_loss / len(train_loader):.4f}")

    model.eval()
    bert_pred = []
    with torch.no_grad():
        for input_ids, attention_mask, _ in test_loader:
            logits = model(
                input_ids=input_ids.to(device), attention_mask=attention_mask.to(device)
            ).logits
            bert_pred.extend(logits.argmax(dim=-1).cpu().tolist())

    evaluate("AraBERT (fine-tuned)", bert_pred)
else:
    print("Skipped (RUN_ARABERT = False)")

## 7. Comparison of all models

Ranked by macro-F1, which weights the three classes equally. Compare each row with the majority-class baseline:
a model that does not clearly beat it has not learned much beyond the class imbalance.

In [ ]:
summary = (
    pd.DataFrame({name: {k: v for k, v in r.items() if k != "y_pred"} for name, r in RESULTS.items()})
    .T.astype(float)
    .sort_values("macro_f1", ascending=False)
    .round(3)
)
summary

## 8. Best model in detail

In [ ]:
best_name = summary.index[0]
best_pred = RESULTS[best_name]["y_pred"]
print(f"Best model by macro-F1: {best_name}\n")
print(classification_report(y_test, best_pred, target_names=CLASS_NAMES, zero_division=0))

plt.figure(figsize=(5, 4))
sns.heatmap(
    confusion_matrix(y_test, best_pred), annot=True, fmt="d", cmap="Blues", cbar=False,
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title(best_name)
plt.tight_layout()
plt.show()